**Load Libraries**

In [18]:
import cv2
import numpy as np

from rembg import remove
from PIL import Image
import os
import random

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model


**Image Preprocessing**

In [2]:
def preprocess(path, IMG_SIZE = 224):

    input_image = Image.open(path)
    output_image = remove(input_image)

    output_image = output_image.convert("RGB")

    img = np.array(output_image)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    img = img / 255.0

    return img

**Setup Dataset**

In [3]:
def create_pairs(dataset_path):

    persons = os.listdir(dataset_path)

    img1 = []
    img2 = []
    labels = []

    for person in persons:

        person_path = os.path.join(dataset_path,person)
        images = os.listdir(person_path)

        # positive pairs
        for i in range(len(images)-1):

            img_a = preprocess(os.path.join(person_path,images[i]))
            img_b = preprocess(os.path.join(person_path,images[i+1]))

            img1.append(img_a)
            img2.append(img_b)
            labels.append(1)

        # negative pairs
        other_person = random.choice(persons)

        if other_person != person:

            other_path = os.path.join(dataset_path,other_person)
            other_images = os.listdir(other_path)

            img_a = preprocess(os.path.join(person_path,images[0]))
            img_b = preprocess(os.path.join(other_path,other_images[0]))

            img1.append(img_a)
            img2.append(img_b)
            labels.append(0)

    return np.array(img1),np.array(img2),np.array(labels)

**Build Model**

In [20]:
from tensorflow.keras.layers import Layer

class L1DistanceLayer(Layer):
    def call(self, inputs):
        x, y = inputs
        return tf.abs(x - y)

In [21]:
def build_siamese():

    base = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )

    for layer in base.layers:
        layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512,activation="relu")(x)
    x = layers.Dense(128,activation="relu")(x)

    feature_extractor = Model(base.input,x)

    inputA = tf.keras.Input(shape=(224,224,3))
    inputB = tf.keras.Input(shape=(224,224,3))

    featA = feature_extractor(inputA)
    featB = feature_extractor(inputB)

    distance = L1DistanceLayer()([featA, featB])

    output = layers.Dense(1,activation="sigmoid")(distance)

    model = Model([inputA,inputB],output)

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

**Model Training**

In [22]:
dataset = "Dataset"

img1,img2,labels = create_pairs(dataset)

X1_train,X1_test,X2_train,X2_test,y_train,y_test = train_test_split(
    img1,img2,labels,test_size=0.2
)

model = build_siamese()

model.fit(
    [X1_train,X2_train],
    y_train,
    validation_data=([X1_test,X2_test],y_test),
    epochs=10,
    batch_size=16
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.7500 - loss: 0.6913 - val_accuracy: 0.0000e+00 - val_loss: 0.6995
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6669 - val_accuracy: 0.0000e+00 - val_loss: 0.6995
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6451 - val_accuracy: 0.0000e+00 - val_loss: 0.7011
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6269 - val_accuracy: 0.0000e+00 - val_loss: 0.7042
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6081 - val_accuracy: 0.0000e+00 - val_loss: 0.7088
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.5893 - val_accuracy: 0.0000e+00 - val_loss: 0.7128
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.5722 - val_accuracy: 0.0000e+00 - val_loss: 0.7164
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.5546 - val_accuracy: 0.0000e+00 - v

**Save Model**

In [23]:
model.save("Models/signature_siamese_model.keras")

**Accuracy Test**

In [24]:
loss,accuracy = model.evaluate([X1_test,X2_test],y_test)

print("Test Accuracy:",accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.0000e+00 - loss: 0.7330
Test Accuracy: 0.0


**Get Prediction using saved Model**

In [25]:
model = load_model(
    "Models/signature_siamese_model.keras",
    custom_objects={"L1DistanceLayer": L1DistanceLayer}
)

In [29]:
img1 = preprocess(r"Dataset\Person1\1.png")
img2 = preprocess(r"Dataset\Person3\2.png")

img1 = np.expand_dims(img1,axis=0)
img2 = np.expand_dims(img2,axis=0)

score = model.predict([img1,img2])[0][0]

print("Similarity Score:",score)

if score > 0.5:
    print("Signature Verified")
else:
    print("Signature Forged")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step
Similarity Score: 0.28598532
Signature Forged
